# 决策树第四课：ID3 决策树

本课目标：不是背 ID3，而是把它每一步怎么选特征、怎么分裂、怎么继续建树都算清楚。

一句话版：**ID3 在每个节点都选择信息增益最大的特征。**

## 1. ID3 的工作流程

ID3 是分类决策树算法，属于有监督学习。训练数据里必须有特征和标签。

它在每个节点重复做这几件事：

1. 看当前节点的数据有多混乱，计算 $H(D)$。
2. 分别尝试每一个候选特征。
3. 对每个特征计算条件熵 $H(D|A)$。
4. 计算信息增益 $Gain(D,A)=H(D)-H(D|A)$。
5. 选择信息增益最大的特征来分裂。
6. 对每个分支里的子数据，继续重复上面的过程。

所以 ID3 的核心问题是：**当前先问哪个问题，能让结果最不混乱？**

## 2. 例子一：是否打球

继续使用上一课的数据：

| 编号 | 天气 | 湿度 | 是否打球 |
| ---: | --- | --- | --- |
| 1 | 晴 | 高 | 否 |
| 2 | 晴 | 高 | 否 |
| 3 | 晴 | 正常 | 是 |
| 4 | 晴 | 正常 | 是 |
| 5 | 晴 | 高 | 是 |
| 6 | 晴 | 正常 | 是 |
| 7 | 雨 | 高 | 否 |
| 8 | 雨 | 正常 | 否 |
| 9 | 雨 | 高 | 否 |
| 10 | 雨 | 正常 | 是 |

上一课已经算过：

- `Gain(D, 天气) = 0.1245`
- `Gain(D, 湿度) = 0.2781`

所以 ID3 会先选择 `湿度` 作为根节点。

In [ ]:
from collections import Counter, defaultdict
from math import log2

def entropy(labels):
    total = len(labels)
    counts = Counter(labels)
    result = -sum((count / total) * log2(count / total) for count in counts.values())
    return 0.0 if abs(result) < 1e-12 else result

def split_by_feature(rows, feature):
    groups = defaultdict(list)
    for row in rows:
        groups[row[feature]].append(row)
    return groups

def conditional_entropy(rows, feature, target):
    total = len(rows)
    groups = split_by_feature(rows, feature)
    return sum(
        (len(group_rows) / total) * entropy([row[target] for row in group_rows])
        for group_rows in groups.values()
    )

def information_gain(rows, feature, target):
    labels = [row[target] for row in rows]
    return entropy(labels) - conditional_entropy(rows, feature, target)

def majority_label(rows, target):
    labels = [row[target] for row in rows]
    return Counter(labels).most_common(1)[0][0]

In [ ]:
play_data = [
    {'天气': '晴', '湿度': '高', '是否打球': '否'},
    {'天气': '晴', '湿度': '高', '是否打球': '否'},
    {'天气': '晴', '湿度': '正常', '是否打球': '是'},
    {'天气': '晴', '湿度': '正常', '是否打球': '是'},
    {'天气': '晴', '湿度': '高', '是否打球': '是'},
    {'天气': '晴', '湿度': '正常', '是否打球': '是'},
    {'天气': '雨', '湿度': '高', '是否打球': '否'},
    {'天气': '雨', '湿度': '正常', '是否打球': '否'},
    {'天气': '雨', '湿度': '高', '是否打球': '否'},
    {'天气': '雨', '湿度': '正常', '是否打球': '是'},
]

play_target = '是否打球'
play_features = ['天气', '湿度']

for feature in play_features:
    print(feature, information_gain(play_data, feature, play_target))

## 3. 例子二：贷款审批

这次我们完整走一遍 ID3 的每一步。

任务：根据用户是否有工作、是否有房产，判断是否批准贷款。

| 编号 | 有工作 | 有房产 | 是否批准贷款 |
| ---: | --- | --- | --- |
| 1 | 是 | 是 | 批准 |
| 2 | 否 | 是 | 批准 |
| 3 | 是 | 否 | 批准 |
| 4 | 是 | 否 | 批准 |
| 5 | 否 | 否 | 拒绝 |
| 6 | 否 | 否 | 拒绝 |
| 7 | 是 | 是 | 批准 |
| 8 | 否 | 否 | 拒绝 |

候选特征只有两个：

- `有工作`
- `有房产`

标签是：`是否批准贷款`。

## 4. 第一步：先算整个数据集的信息熵

全部 8 条数据里：

- 批准：5 条
- 拒绝：3 条

所以：

$$P(批准)=\frac{5}{8}=0.625$$

$$P(拒绝)=\frac{3}{8}=0.375$$

代入信息熵公式：

$$H(D)=-(0.625\log_2 0.625+0.375\log_2 0.375)$$

分开算两项：

$$\log_2 0.625 \approx -0.6781$$

$$0.625\times(-0.6781) \approx -0.4238$$

$$\log_2 0.375 \approx -1.4150$$

$$0.375\times(-1.4150) \approx -0.5306$$

所以：

$$H(D)=-(-0.4238-0.5306)=0.9544$$

这就是分裂前的混乱程度。

## 5. 第二步：尝试特征“有工作”

先按 `有工作` 把 8 条数据分成两组。

| 有工作 | 样本编号 | 样本数 | 批准 | 拒绝 |
| --- | --- | ---: | ---: | ---: |
| 是 | 1, 3, 4, 7 | 4 | 4 | 0 |
| 否 | 2, 5, 6, 8 | 4 | 1 | 3 |

先算 `有工作 = 是` 这一组。

这一组 4 条全是批准：

$$P(批准)=\frac{4}{4}=1$$

$$P(拒绝)=\frac{0}{4}=0$$

一个组如果全是同一类，就完全不混乱，熵为 0：

$$H(有工作=是)=0$$

## 6. 继续算“有工作 = 否”这一组

`有工作 = 否` 这一组有 4 条：1 条批准，3 条拒绝。

所以：

$$P(批准)=\frac{1}{4}=0.25$$

$$P(拒绝)=\frac{3}{4}=0.75$$

代入熵公式：

$$H(有工作=否)=-(0.25\log_2 0.25+0.75\log_2 0.75)$$

分开算：

$$\log_2 0.25=-2$$

$$0.25\times(-2)=-0.5$$

$$\log_2 0.75 \approx -0.4150$$

$$0.75\times(-0.4150) \approx -0.3113$$

所以：

$$H(有工作=否)=-(-0.5-0.3113)=0.8113$$

## 7. 得到“有工作”的条件熵和信息增益

现在把两个分组的熵按样本数加权平均。

- `有工作 = 是`：4 条，占全部的 $\frac{4}{8}$，组内熵 0
- `有工作 = 否`：4 条，占全部的 $\frac{4}{8}$，组内熵 0.8113

所以：

$$H(D|有工作)=\frac{4}{8}\times0+\frac{4}{8}\times0.8113$$

$$H(D|有工作)=0+0.4056=0.4056$$

信息增益：

$$Gain(D, 有工作)=H(D)-H(D|有工作)$$

$$Gain(D, 有工作)=0.9544-0.4056=0.5488$$

这句话翻译成人话就是：**问“有没有工作”以后，不确定性从 0.9544 降到 0.4056，减少了 0.5488。**

## 8. 第三步：尝试特征“有房产”

现在换一个候选特征，按 `有房产` 分组。

| 有房产 | 样本编号 | 样本数 | 批准 | 拒绝 |
| --- | --- | ---: | ---: | ---: |
| 是 | 1, 2, 7 | 3 | 3 | 0 |
| 否 | 3, 4, 5, 6, 8 | 5 | 2 | 3 |

`有房产 = 是` 这一组 3 条全是批准，所以：

$$H(有房产=是)=0$$

`有房产 = 否` 这一组有 5 条：2 条批准，3 条拒绝。

所以：

$$P(批准)=\frac{2}{5}=0.4$$

$$P(拒绝)=\frac{3}{5}=0.6$$

## 9. 继续算“有房产 = 否”的组内熵

代入熵公式：

$$H(有房产=否)=-(0.4\log_2 0.4+0.6\log_2 0.6)$$

分开算：

$$\log_2 0.4 \approx -1.3219$$

$$0.4\times(-1.3219) \approx -0.5288$$

$$\log_2 0.6 \approx -0.7370$$

$$0.6\times(-0.7370) \approx -0.4422$$

所以：

$$H(有房产=否)=-(-0.5288-0.4422)=0.9710$$

## 10. 得到“有房产”的条件熵和信息增益

- `有房产 = 是`：3 条，占全部的 $\frac{3}{8}$，组内熵 0
- `有房产 = 否`：5 条，占全部的 $\frac{5}{8}$，组内熵 0.9710

所以：

$$H(D|有房产)=\frac{3}{8}\times0+\frac{5}{8}\times0.9710$$

$$H(D|有房产)=0+0.6069=0.6069$$

信息增益：

$$Gain(D, 有房产)=0.9544-0.6069=0.3475$$

因为四舍五入位置不同，代码里可能显示 `0.3476`，本质是同一个数。

## 11. 第四步：比较两个特征，选根节点

现在把两条候选路线放在一起：

| 候选特征 | 分裂前熵 | 条件熵 | 信息增益 |
| --- | ---: | ---: | ---: |
| 有工作 | 0.9544 | 0.4056 | 0.5488 |
| 有房产 | 0.9544 | 0.6069 | 0.3475 |

`有工作` 的信息增益更大，所以 ID3 第一刀选择：

```text
有工作?
```

这就是根节点。

## 12. 第五步：继续处理每个分支

根节点选了 `有工作` 以后，树先长成这样：

```text
有工作?
├── 是
└── 否
```

现在分别看两个分支。

`有工作 = 是` 的分支里有 4 条数据，全部是批准：

| 编号 | 有工作 | 有房产 | 是否批准贷款 |
| ---: | --- | --- | --- |
| 1 | 是 | 是 | 批准 |
| 3 | 是 | 否 | 批准 |
| 4 | 是 | 否 | 批准 |
| 7 | 是 | 是 | 批准 |

这个分支已经纯了，所以直接变成叶子：

```text
有工作?
├── 是 -> 批准
└── 否
```

## 13. 继续处理“有工作 = 否”的分支

`有工作 = 否` 的分支里有 4 条数据：

| 编号 | 有工作 | 有房产 | 是否批准贷款 |
| ---: | --- | --- | --- |
| 2 | 否 | 是 | 批准 |
| 5 | 否 | 否 | 拒绝 |
| 6 | 否 | 否 | 拒绝 |
| 8 | 否 | 否 | 拒绝 |

这个分支还不纯：1 个批准，3 个拒绝。

但此时还剩下一个没用过的特征：`有房产`。

按 `有房产` 再分：

| 有房产 | 样本编号 | 批准 | 拒绝 | 结果 |
| --- | --- | ---: | ---: | --- |
| 是 | 2 | 1 | 0 | 纯，批准 |
| 否 | 5, 6, 8 | 0 | 3 | 纯，拒绝 |

所以最终树是：

```text
有工作?
├── 是 -> 批准
└── 否 -> 有房产?
          ├── 是 -> 批准
          └── 否 -> 拒绝
```

In [ ]:
loan_data = [
    {'有工作': '是', '有房产': '是', '是否批准贷款': '批准'},
    {'有工作': '否', '有房产': '是', '是否批准贷款': '批准'},
    {'有工作': '是', '有房产': '否', '是否批准贷款': '批准'},
    {'有工作': '是', '有房产': '否', '是否批准贷款': '批准'},
    {'有工作': '否', '有房产': '否', '是否批准贷款': '拒绝'},
    {'有工作': '否', '有房产': '否', '是否批准贷款': '拒绝'},
    {'有工作': '是', '有房产': '是', '是否批准贷款': '批准'},
    {'有工作': '否', '有房产': '否', '是否批准贷款': '拒绝'},
]

loan_target = '是否批准贷款'
loan_features = ['有工作', '有房产']

print(f"贷款数据 H(D) = {entropy([row[loan_target] for row in loan_data]):.4f}\n")

for feature in loan_features:
    groups = split_by_feature(loan_data, feature)
    print(f"尝试特征：{feature}")
    for value, rows in groups.items():
        labels = [row[loan_target] for row in rows]
        weight = len(rows) / len(loan_data)
        group_entropy = entropy(labels)
        print(f"  {feature}={value}: {len(rows)}条, 标签统计{dict(Counter(labels))}")
        print(f"    组内熵 = {group_entropy:.4f}")
        print(f"    权重 = {len(rows)}/{len(loan_data)} = {weight:.3f}")
        print(f"    加权贡献 = {weight * group_entropy:.4f}")
    cond = conditional_entropy(loan_data, feature, loan_target)
    gain = information_gain(loan_data, feature, loan_target)
    print(f"  条件熵 H(D|{feature}) = {cond:.4f}")
    print(f"  信息增益 Gain(D,{feature}) = {gain:.4f}\n")

In [ ]:
def build_id3_tree(rows, features, target):
    labels = [row[target] for row in rows]

    if len(set(labels)) == 1:
        return labels[0]

    if not features:
        return majority_label(rows, target)

    best_feature = max(features, key=lambda f: information_gain(rows, f, target))
    tree = {best_feature: {}}
    remaining_features = [f for f in features if f != best_feature]

    for feature_value, group_rows in split_by_feature(rows, best_feature).items():
        tree[best_feature][feature_value] = build_id3_tree(
            group_rows,
            remaining_features,
            target
        )

    return tree

loan_tree = build_id3_tree(loan_data, loan_features, loan_target)
loan_tree

## 14. 本课小结

ID3 建树的每一步都可以拆成这种固定流程：

1. 先算当前数据集的熵。
2. 对每个候选特征分组。
3. 算每个分组内部的熵。
4. 用分组样本占比做加权平均，得到条件熵。
5. 用 `分裂前熵 - 条件熵` 得到信息增益。
6. 选信息增益最大的特征。
7. 对每个分支继续重复，直到分支纯了或者没有特征可用。

你真正要抓住的是：**ID3 不是一次算完整棵树，而是在每个节点局部地选当前最能降低混乱的特征。**